<a href="https://colab.research.google.com/github/PandaX187/OcuTriage/blob/main/OcuTriage_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Updated path based on your sidebar screenshot
!unzip -q "/content/drive/MyDrive/OcuTriage_Project/Dataset/train_images.zip" -d "/content/dataset"

In [ ]:
import pandas as pd
import os
import shutil

# 1. Set your paths based on your sidebar screenshot
# We use the folder inside OcuTriage_Project where your CSV is
base_path = '/content/drive/MyDrive/OcuTriage_Project/Dataset/'
csv_path = os.path.join(base_path, 'train.csv')
unzipped_images = '/content/dataset/train_images/' # This is where we just unzipped them

# 2. Create the Traffic Light folders in your Drive
categories = ['Green_No_DR', 'Yellow_Mild_Moderate', 'Red_Severe_Proliferative']
for cat in categories:
    os.makedirs(os.path.join(base_path, cat), exist_ok=True)

# 3. Load the labels
df = pd.read_csv(csv_path)

print("Starting to sort images into Traffic Light categories...")

# 4. Loop through and copy files
for index, row in df.iterrows():
    img_name = row['id_code'] + '.png'
    diagnosis = row['diagnosis']

    # Traffic Light Logic based on severity grades 0-4
    if diagnosis == 0:
        dest_folder = categories[0] # Green
    elif diagnosis in [1, 2]:
        dest_folder = categories[1] # Yellow
    else:
        dest_folder = categories[2] # Red

    src = os.path.join(unzipped_images, img_name)
    dst = os.path.join(base_path, dest_folder, img_name)

    if os.path.exists(src):
        shutil.copy(src, dst)

print("✅ Task 1.1 Complete! Check your Google Drive 'Dataset' folder.")

Starting to sort images into Traffic Light categories...
✅ Task 1.1 Complete! Check your Google Drive 'Dataset' folder.


In [ ]:

!pip install split-folders

import splitfolders


input_folder = '/content/drive/MyDrive/OcuTriage_Project/Dataset/'
output_folder = '/content/drive/MyDrive/OcuTriage_Project/Final_Split_Data/'

# Split with a ratio of (train, val, test)
# 0.8 = 80%, 0.1 = 10%, 0.1 = 10%
splitfolders.ratio(input_folder, output=output_folder, seed=1337, ratio=(.8, .1, .1), group_prefix=None)

Copying files: 3662 files [10:03,  6.07 files/s]


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

# 1. Mount and Load Data (Make sure paths are correct)
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/drive/MyDrive/OcuTriage_Project/Final_Split_Data/train',
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/drive/MyDrive/OcuTriage_Project/Final_Split_Data/val',
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

# 2. Build EfficientNet-B0 with Transfer Learning
# This is fast because we 'freeze' the base layers
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.2), # Prevents the model from 'memorizing'
    layers.Dense(3, activation='softmax') # Green, Yellow, Red
])

# 3. Compile and Run
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Start with 10 Epochs to see the progress
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

Found 2929 files belonging to 3 classes.
Found 364 files belonging to 3 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 750s 8s/step - accuracy: 0.7644 - loss: 0.5820 - val_accuracy: 0.8187 - val_loss: 0.5104
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 248s 3s/step - accuracy: 0.8395 - loss: 0.3923 - val_accuracy: 0.8626 - val_loss: 0.4044
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 261s 3s/step - accuracy: 0.8607 - loss: 0.3602 - val_accuracy: 0.8819 - val_loss: 0.3443
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 246s 3s/step - accuracy: 0.8607 - loss: 0.3322 - val_accuracy: 0.8846 - val_loss: 0.3024
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 279s 3s/step - accuracy: 0.8836 - loss: 0.2971 - val_accuracy: 0.8901 - val_loss: 0.2908
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 250s 3s/step - accuracy: 0.8846 - loss: 0.3012 - val_accuracy: 0.8874 - val_loss: 0.2787
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 262s 3s/step - accuracy: 0.8849 - loss: 0.2950 - val_accuracy: 0.890

In [4]:
model.save('/content/drive/MyDrive/OcuTriage_Project/ocutriage_model_v1.keras')
print("Native Keras model saved as well!")

Native Keras model saved as well!
